*Cellule 1 — Vérifier l'environnement*

In [ ]:
import openai, pandas as pd


print("openai version:", openai.__version__)
print("pandas version:", pd.__version__)

*Cellule 2 — Connexion au serveur*

In [ ]:
from openai import OpenAI

client = OpenAI(base_url="http://localhost:1234/v1", api_key="lm-studio", timeout=120.0)
print("Modèles disponibles :")
for m in client.models.list().data:
    print(" -", m.id)

*Cellule 3 — Un tout premier échange*

In [ ]:
response = client.chat.completions.create(
    model="mistralai/mistral-7b-instruct-v0.3",
    messages=[{"role": "user", "content": "Say hello in one word."}],
)
print("Réponse :", response.choices[0].message.content)

*Cellule 4 — Un prompt un peu plus long (mi-chemin)*

In [ ]:
messages_system = [
    {"role": "system", "content": "You are a supply chain analyst. Be concise and factual."},
    {"role": "user", "content": "A supplier has 15 delivery delays this month and 3 quality issues. Summarize in one sentence."},
]

for model in ["mistralai/mistral-7b-instruct-v0.3", "google/gemma-4-e4b"]:
    try:
        r = client.chat.completions.create(
            model=model, messages=messages_system, temperature=0.2, max_tokens=100
        )
        print(f"[OK]   {model} : {r.choices[0].message.content.strip()}")
    except Exception as e:
        print(f"[FAIL] {model} : {str(e)[:120]}")

*Cellule 5 — Charger tes données*

In [ ]:
import sys
from pathlib import Path
import pandas as pd

cwd = Path.cwd()
SCRIPTS_DIR = cwd if cwd.name == "scripts" else cwd / "scripts"
DATA_PATH = SCRIPTS_DIR.parent / "data" / "cleaned_comments.csv"

if str(SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_DIR))

print("Dossier scripts :", SCRIPTS_DIR)
print("Fichier de données existe :", DATA_PATH.exists())

df = pd.read_csv(DATA_PATH)
print("Lignes :", len(df))
print("Colonnes clés présentes :", all(c in df.columns for c in ["Comment_EN", "Root_Cause", "Snapshot_Date"]))

In [ ]:
from pathlib import Path

root = Path.home() / "Roche_AI_PoC"
print("Dossier courant :", Path.cwd())
print("\nContenu de data/ :")
for p in sorted((root / "data").iterdir()):
    print("  ", p.name)

print("\nRecherche de cleaned_comments* dans tout le projet (hors venv) :")
for p in root.rglob("cleaned_comments*"):
    if "venv" not in p.parts:
        print("  ", p, f"({p.stat().st_size/1e6:.1f} Mo)")

*Cellule 6 — Calculer les stats*

In [ ]:
print(df.columns.tolist())
print("Comment_EN présent :", "Comment_EN" in df.columns)

In [ ]:
from stats_engine import compute_stats

stats = compute_stats(df)
print("total_comments :", stats["total_comments"])
print("Clés :", list(stats.keys()))

*Cellule 7 — Construire le prompt (sans appeler le serveur)*

In [ ]:
import importlib, llm_summary
importlib.reload(llm_summary)
from llm_summary import build_prompt

sample_comments = stats["top_comments"]
messages = build_prompt(stats, sample_comments)

print("Nb de messages :", len(messages))
print("Rôle(s) :", [m["role"] for m in messages])
print("Taille (caractères) :", sum(len(m["content"]) for m in messages))
print("\n--- DÉBUT DU PROMPT (600 premiers caractères) ---\n")
print(messages[0]["content"][:600])

*Cellule 8 — L'appel final (le vrai test)*

In [ ]:
import time
import importlib, llm_summary
importlib.reload(llm_summary)            # indispensable après la correction de l'URL
from llm_summary import generate_summary, is_server_available

print("Serveur joignable :", is_server_available())

t0 = time.time()
summary = generate_summary(stats, sample_comments)
print(f"Durée : {time.time() - t0:.1f} s\n")
print("=== AI SUMMARY ===")
print(summary)